In [ ]:
####################################
#ENVIRONMENT SETUP

In [ ]:
#Importing Libraries
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.ticker as ticker
import matplotlib.cm as cm
from matplotlib.colors import Normalize
from matplotlib.ticker import MaxNLocator
from matplotlib.ticker import ScalarFormatter
from matplotlib.lines import Line2D
import matplotlib.gridspec as gridspec
import xarray as xr

import sys; import os; import time; from datetime import timedelta
import pickle
os.environ["HDF5_USE_FILE_LOCKING"] = "FALSE"
import h5py
from tqdm import tqdm

#Specific Libraries
from matplotlib.lines import Line2D
import matplotlib

In [ ]:
#MAIN DIRECTORIES
def GetDirectories():
    mainDirectory='/mnt/lustre/koa/koastore/torri_group/air_directory/Projects/DCI-Project/'
    mainCodeDirectory=os.path.join(mainDirectory,"Code/CodeFiles/")
    scratchDirectory='/mnt/lustre/koa/scratch/air673/'
    codeDirectory=os.getcwd()
    return mainDirectory,mainCodeDirectory,scratchDirectory,codeDirectory

[mainDirectory,mainCodeDirectory,scratchDirectory,codeDirectory] = GetDirectories()

In [ ]:
#IMPORT CLASSES
sys.path.append(os.path.join(mainCodeDirectory,"2_Variable_Calculation"))
from CLASSES_Variable_Calculation import ModelData_Class, SlurmJobArray_Class, DataManager_Class

#IMPORT FUNCTIONS (from current directory)
sys.path.append(os.path.join(mainCodeDirectory,"2_Variable_Calculation"))
from FUNCTIONS_Variable_Calculation import *

sys.path.append(os.path.join(mainDirectory, "Code", "CodeFiles", "Functions", "Classes"))
from DataPlotting_Classes import DataPlotting_Classes
UltimatePlotting = DataPlotting_Classes.UltimatePlotting_Class

In [ ]:
####################################
#LOADING CLASSES

In [ ]:
#data loading class
ModelData = ModelData_Class(mainDirectory, scratchDirectory, simulationNumber=6)
#data manager class
DataManager = DataManager_Class(mainDirectory, scratchDirectory, ModelData, dataType="Figures", dataName="CloudTopFigure", dtype="float32",make_dirs=False)

In [ ]:
####################################
#DATA LOADING

In [ ]:
# SUBSETTING SBF and COLDPOOL PARCELS

#Functions
#getting convergence xmax
def Get_AvgConvergence(t):

    timeString = ModelData.timeStrings[t]
    outputDataDirectory=os.path.normpath(os.path.join(DataManager_TrackedParcels.outputDataDirectory,"..","Eulerian_CLTracking"))
    Dictionary = TrackingAlgorithms_DataLoading_Class.LoadData(ModelData, DataManager_TrackedParcels, timeString,
                     dataName="Eulerian_CLTracking",outputDataDirectory=outputDataDirectory,printstatement=False)
    avgConvergence = Dictionary["avgConvergence"]
    return avgConvergence

def find_SBF_xmaxs():
    xmaxs=[]
    for t in tqdm(range(ModelData.Ntime)):
        if t == 0:
            xmaxs.append(np.nan)
        else:
            avgConvergence = Get_AvgConvergence(t)
            avgConvergence_max=np.nanmax(avgConvergence)
            xmax = np.where(avgConvergence==avgConvergence_max)[0][0]
            xmaxs.append(xmax)
    return xmaxs

def Get_SBF_xmaxs_FilePath(ModelData, DataManager):
    fileName = (
        f"SBF_xmaxs_{ModelData.res}_{ModelData.t_res}_"
        f"{ModelData.Ntime}nt.pkl"
    )
    return os.path.join(DataManager.outputDataDirectory, fileName)

def LoadOrRun_find_SBF_xmaxs(ModelData, DataManager, overwrite=False):
    filePath = Get_SBF_xmaxs_FilePath(ModelData, DataManager)
    
    if os.path.exists(filePath) and not overwrite:
        print(f"reading from {filePath}")
        with open(filePath, "rb") as f:
            xmaxs = pickle.load(f)
        return xmaxs

    xmaxs = find_SBF_xmaxs()

    os.makedirs(os.path.dirname(filePath), exist_ok=True)
    with open(filePath, "wb") as f:
        pickle.dump(xmaxs, f)
    print(f"saved to {filePath}")
    return xmaxs

#Importing Classes
sys.path.append(os.path.join(mainCodeDirectory,"3_Project_Algorithms","2_Tracking_Algorithms"))
from CLASSES_TrackingAlgorithms import TrackingAlgorithms_DataLoading_Class

DataManager_TrackedParcels = DataManager_Class(mainDirectory, scratchDirectory, ModelData, dataType="Tracking_Algorithms", dataName="Lagrangian_UpdraftTracking",dtype='float32',codeSection = "Project_Algorithms")

#Calculating
try:
    xmaxs = LoadOrRun_find_SBF_xmaxs(ModelData, DataManager_TrackedParcels)
except:
    xmaxs = [ModelData.Nxh]*ModelData.Ntime

In [ ]:
####################################
#CALCULATION FUNCTIONS

In [ ]:
#Calculation Functions
def CalculateCloudTopArray(meanMax):
    if ModelData.Nzh==34:
        allData=ModelData.OpenData()
    
    cloud_top_arr = np.full((ModelData.Ntime, len(xh)), np.nan)
    
    times=np.arange(0, ModelData.Ntime)
    for t in tqdm(times):
        if ModelData.Nzh==34:
            data=allData.isel(time=t) 
        else:
            data=ModelData.OpenData_SingleTime(t=t).isel(time=0)
        qcqi = data['qc']+data['qi']
        
        a = qcqi.data
        a[a <= 1e-6] = np.nan
        
        valid_mask = ~np.isnan(a)  # shape (z, x)
        top_idx = np.argmax(valid_mask[::-1, :], axis=0)
        top_idx = (valid_mask.shape[0] - 1) - top_idx
        cloud_top = np.where(valid_mask.any(axis=0), ModelData.zh[top_idx], np.nan)

        if meanMax == "Mean":
            cloud_top_arr[t, :] = np.nanmean(cloud_top,axis=0)
        elif meanMax == "Max":
            cloud_top_arr[t, :] = np.nanmax(cloud_top,axis=0)
        elif meanMax == "Percentile95":
            cloud_top_arr[t, :] = np.nanpercentile(cloud_top,95,axis=0)
    return cloud_top_arr

def CalculateCloudBaseArray(meanMax, maxBase_km=4.0, threshold=1e-6):

    xh = ModelData.xh - ModelData.xh[0]
    if ModelData.Nzh == 34:
        allData = ModelData.OpenData()

    cloud_base_arr = np.full((ModelData.Ntime, len(xh)), np.nan)

    for t in tqdm(np.arange(0, ModelData.Ntime)):
        if ModelData.Nzh == 34:
            data = allData.isel(time=t)
        else:
            data = ModelData.OpenData_SingleTime(t=t).isel(time=0)
        qcqi = data['qc']# + data['qi']

        a = np.array(qcqi.data)                    # (z, y, x)
        valid_mask = a >= threshold                # cloudy points

        # lowest cloudy level in each (y, x) column
        base_idx = np.argmax(valid_mask, axis=0)
        cloud_base = np.where(valid_mask.any(axis=0), ModelData.zh[base_idx], np.nan)

        # keep only bases below the cutoff (zh is in km)
        cloud_base[cloud_base > maxBase_km] = np.nan

        # reduce over y
        if meanMax == "Mean":
            cloud_base_arr[t, :] = np.nanmean(cloud_base, axis=0)
        elif meanMax == "Min":
            cloud_base_arr[t, :] = np.nanmin(cloud_base, axis=0)
        elif meanMax == "Percentile5":
            cloud_base_arr[t, :] = np.nanpercentile(cloud_base, 5, axis=0)
    return cloud_base_arr

xh = ModelData.xh-ModelData.xh[0]

def IndexWithXMaxs(xh,xmaxs,
                   removeOutliers=True):
    xmaxs=xmaxs.copy()
    xmaxs[0]=xmaxs[1]
    
    xmax_locations = xh[np.array(xmaxs).astype(int)]
    
    xmax_locations[:np.abs(ModelData.time_hrs-10).argmin()]=np.nan

    if removeOutliers:
        # remove anything between time_hrs 10 and 11 that is > 200 (for plotting convenience)
        # (replace with nearest value to the left instead of NaN)
        t_start = np.abs(ModelData.time_hrs-10).argmin()
        t_end = np.abs(ModelData.time_hrs-11).argmin()
        idx = np.arange(len(xmax_locations))
        mask = (idx >= t_start) & (idx <= t_end) & (xmax_locations > 200)
    
        for i in np.where(mask)[0]:
            xmax_locations[i] = xmax_locations[i-1]  # nearest value to the left
            
    return xmax_locations


In [ ]:
####################################
#PLOTTING FUNCTIONS

In [ ]:
#Combined Plot
def MakeCloudTopFigure_V1(cloud_top_arr, fontScale=1.4, figSize=(12, 10), tickSize=20,
                       tMin=10, tMax=17, lineStepHrs=0.5):
    [fs] = UltimatePlotting.GetFontStyles(fontScale)

    [fig, axes, caxes] = UltimatePlotting.MakeAlignedGridFigure(
        nRows=2, nCols=1, figSize=figSize,
        hasCbar=[[True], [True]], cbarSide="right",
        rowGapRatio=0.08, cbarPaddingRatioRight=0.02, cbarThicknessRatio=0.03,
        shareX=True)
    
    xLim = (125, 400) #xLim = (xh[-1] * 0.25, xh[-1] * 0.8)
    time = np.arange(ModelData.Ntime) * ModelData.dt / 3600 + 6
    sbf = IndexWithXMaxs(xh, xmaxs)
    ctLabel = r'$CTH$ (km)'

    # ---------------- top: time-x contour ----------------
    ax = axes[0, 0]
    [ticks, lim, levels] = UltimatePlotting.GenerateTicksAndLim(cloud_top_arr, nTicks=7, nLevels=13)
    UltimatePlotting.PlotContour(
        ax=ax, cax=caxes[0, 0], xData=xh, yData=time, zData=cloud_top_arr,
        cmap='viridis', levels=levels, cbarTicks=ticks, cbarLabel=ctLabel,
        yLabel='Local time (LT)', xLim=xLim, yLim=(tMin, tMax),
        fontScale=fontScale)

    [yTicksT, _, _] = UltimatePlotting.GenerateTicksAndLim([tMin, tMax], nTicks=8)
    ax.set_yticks(yTicksT[(yTicksT >= tMin) & (yTicksT <= tMax)])

    ax.contour(xh, time, np.nan_to_num(cloud_top_arr) >= 6, levels=[0.5], colors='red')
    ax.contour(xh, time, np.nan_to_num(cloud_top_arr, nan=99) <= 4, levels=[0.5], colors='white')
    ax.plot(sbf, time, color='black', linewidth=2)
    ax.axvline(128, color='black', zorder=10)

    handles = [Line2D([0], [0], color='red',   label=r'$CTH \geq 6$ km'),
               Line2D([0], [0], color='white', label=r'$CTH \leq 4$ km'),
               Line2D([0], [0], color='black', lw=2, label='Sea breeze front')]
    leg = ax.legend(handles=handles, loc='lower right', fontsize=fs["legend"]["fontsize"])
    leg.set_zorder(20)

    # ---------------- bottom: CT profiles colored by local time ----------------
    ax = axes[1, 0]
    cmap = plt.cm.viridis_r

    # nice ticks and band edges from the library
    [tTicks, tLim, tLevels] = UltimatePlotting.GenerateTicksAndLim([tMin, tMax], nTicks=8, nLevels=15)
    tTicks = tTicks[(tTicks >= tMin) & (tTicks <= tMax)]
    norm = matplotlib.colors.BoundaryNorm(tLevels, cmap.N)

    # one line (and SBF position) per chosen time
    for tt in np.arange(tMin, tMax + 1e-6, lineStepHrs):
        t = np.abs(time - tt).argmin()
        ax.plot(xh, cloud_top_arr[t, :], color=cmap(norm(tt)), alpha=0.8)
        # ax.axvline(sbf[t], color=cmap(norm(tt)), alpha=0.8, linewidth=1.5)

    ax.axhline(6, color='red', zorder=10); ax.axhline(4, color='red', zorder=10)
    ax.axvline(128, color='black', zorder=10)
    inView = (xh >= xLim[0]) & (xh <= xLim[1])
    [yTicksB, yLimB, _] = UltimatePlotting.GenerateTicksAndLim(cloud_top_arr[:, inView], nTicks=10)
    ax.set_xlim(xLim); ax.set_yticks(yTicksB); ax.set_ylim(0, yLimB[1])
    ax.set_xticks(np.arange(125, 426, 50))
    ax.set_xlabel('x (km)'); ax.set_ylabel(ctLabel)

    sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
    cb = fig.colorbar(sm, cax=caxes[1, 0], ticks=tTicks)
    cb.set_label('Local time (LT)')

    # fonts, then force tick label sizes last
    UltimatePlotting.ApplyFontStyles(ax, fs)
    UltimatePlotting.ApplyColorbarFontStyles(cb, fs)
    for ax_ in axes.ravel():
        ax_.tick_params(axis='both', which='major', labelsize=tickSize)
    for cax_ in caxes.ravel():
        cax_.tick_params(labelsize=tickSize)
    return fig

def MakeCloudTopFigure_V2(cth_arr, cbh_arr, fontScale=1.4, figSize=(12, 10), tickSize=20,
                          tMin=10, tMax=17, xLim=(125, 400), plotType='contourf'):
    [fs] = UltimatePlotting.GetFontStyles(fontScale)

    [fig, axes, caxes] = UltimatePlotting.MakeAlignedGridFigure(
        nRows=2, nCols=1, figSize=figSize,
        hasCbar=[[True], [True]], cbarSide="right",
        rowGapRatio=0.08, cbarPaddingRatioRight=0.02, cbarThicknessRatio=0.03,
        shareX=True)

    xh = ModelData.xh - ModelData.xh[0]
    time = np.arange(ModelData.Ntime) * ModelData.dt / 3600 + 6
    sbf = IndexWithXMaxs(xh, xmaxs)

    panels = [(cth_arr, r'$CTH$ (km)', 'viridis'),
              (cbh_arr, r'$CBH$ (km)', 'viridis')]

    for i, (arr, label, cmap) in enumerate(panels):
        ax = axes[i, 0]
        if i == 1:
            ticks  = np.arange(1, 3 + 1e-6, 0.5)
            levels = np.arange(1, 3 + 1e-6, 0.25)
            extra  = dict(extend='max')
        else:
            ticks  = np.arange(2, 16 + 1e-6, 2)
            levels = np.arange(2, 16 + 1e-6, 1)
            extra  = dict(extend='both')
        
        kw = dict(extra)
        if plotType == 'pcolormesh':
            kw.pop('extend', None)
            kw['shading'] = 'nearest'
        UltimatePlotting.PlotContour(
            ax=ax, cax=caxes[i, 0], xData=xh, yData=time, zData=arr,
            plotType=plotType, cmap=cmap, levels=levels, cbarTicks=ticks, cbarLabel=label,
            yLabel='Local time (LT)', xLim=xLim, yLim=(tMin, tMax),
            xLabel='x (km)' if i == 1 else None,
            fontScale=fontScale, **kw)

        [yTicksT, _, _] = UltimatePlotting.GenerateTicksAndLim([tMin, tMax], nTicks=8)
        ax.set_yticks(yTicksT[(yTicksT >= tMin) & (yTicksT <= tMax)])
        ax.plot(sbf, time, color='black', linewidth=2)
        ax.axvline(128, color='black', zorder=10)

    # threshold contours and legend on the CTH panel only
    ax = axes[0, 0]
    ax.contour(xh, time, np.nan_to_num(cth_arr) >= 6, levels=[0.5], colors='red')
    ax.contour(xh, time, np.nan_to_num(cth_arr, nan=99) <= 4, levels=[0.5], colors='white')
    handles = [Line2D([0], [0], color='red',   label=r'$CTH \geq 6$ km'),
               Line2D([0], [0], color='white', label=r'$CTH \leq 4$ km'),
               Line2D([0], [0], color='black', lw=2, label='Sea breeze front')]
    leg = ax.legend(handles=handles, loc='lower right', fontsize=fs["legend"]["fontsize"])
    leg.set_zorder(20)

    axes[1, 0].set_xticks(np.arange(xLim[0], xLim[1]+25, 25))

    # force tick label sizes last
    for ax_ in axes.ravel():
        ax_.tick_params(axis='both', which='major', labelsize=tickSize)
    for cax_ in caxes.ravel():
        cax_.tick_params(labelsize=tickSize)

    # panel labels to the left of each y axis
    AddLabelLeftOfYAxis(fig, axes[0, 0], "A)", fontsize=tickSize + 4)
    AddLabelLeftOfYAxis(fig, axes[1, 0], "B)", fontsize=tickSize + 4)
    return fig

def AddLabelLeftOfYAxis(fig, ax, label, fontsize=24, pad=0.01):
    """Panel label to the left of the y axis (past tick labels and axis label),
    top-aligned with the panel so it never rises above the axis."""
    fig.canvas.draw()                                    # layout must be final to measure
    renderer = fig.canvas.get_renderer()
    bb = ax.get_tightbbox(renderer).transformed(fig.transFigure.inverted())
    y_top = ax.get_position().y1
    fig.text(bb.x0 - pad, y_top, label, ha='right', va='top',
             fontsize=fontsize, fontweight='normal')

def GetPlottingDirectory(plotFileName, plotType):
    plottingDirectory = mainCodeDirectory=os.path.join(mainDirectory,"Code","PLOTTING")
    
    specificPlottingDirectory = os.path.join(plottingDirectory, plotType, 
                                             f"Simulation_{ModelData.simulationNumber}_{ModelData.res}_{ModelData.t_res}_{ModelData.Nz_str}nz")
    os.makedirs(specificPlottingDirectory, exist_ok=True)

    plottingFileName=os.path.join(specificPlottingDirectory, plotFileName)

    return plottingFileName
def SaveFigure(fig,plotFileTitle):
    plotFileName = f"{plotFileTitle}_{ModelData.res}_{ModelData.t_res}_{ModelData.Np_str}.jpg"
    plottingFileName = GetPlottingDirectory(plotFileName, plotType="Initial_Figures/Figures")
    print(f"Saving Plot to {plottingFileName}")
    fig.savefig(plottingFileName, dpi=300, bbox_inches='tight')
    plt.close(fig)

In [ ]:
####################################
#PLOTTING

In [ ]:
cloud_top_arr = LoadOrRun(CalculateCloudTopArray, DataManager.outputDataDirectory,
                          fileName="CloudTopCalculations_Mean.pkl",
                          kwargs={'meanMax': 'Mean'})

cloud_base_arr = LoadOrRun(CalculateCloudBaseArray, DataManager.outputDataDirectory,
                           fileName="CloudBaseCalculations_Min_below4km.pkl",
                           kwargs={'meanMax': 'Min', 'maxBase_km': 4.0})

fig = MakeCloudTopFigure_V2(cloud_top_arr, cloud_base_arr, fontScale=1.8)
SaveFigure(fig, plotFileTitle="CloudTopFigure_Mean")
fig